In [ ]:
import regex as re

In [ ]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

### Gutenberg data download 

In [ ]:
import os 
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open (file_path, "wb") as f:
        f.write(response.content)

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"total lenght of the text:- {len(raw_text)}")
print(raw_text[:99])

&nbsp;
## 2.1 Understanding word embeddings

In [ ]:
import re


text = "Hello, world. This, is a test."
result = re.split(r"(\s)", text)

print(result)

- We don't only want to split on whitespaces but also commas and periods, so let's modify the regular expression to do that as well

In [ ]:
result = re.split(r"([,.]|\s)", text)
print(result)

#####  👇 this is not used for splitting the words , preserve the whitespace and punctations is not split


In [ ]:
text.split()

In [ ]:
result =  [item for item in result if item.strip()]
print(result)

In [ ]:
text  = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result =  [item.strip() for item in result if item.strip()]
print(result)

In [ ]:
preprocessed =  re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

In [ ]:
print(len(preprocessed))

&nbsp;
## 2.2 Converting tokens into token IDs

In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

In [ ]:
vocab = {token:integer for integer, token in enumerate(all_words)}

In [ ]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i>=50:
        break

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]   
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids =  tokenizer.encode(text)
print(ids)

In [ ]:
print(tokenizer.decode(ids))

In [ ]:
print(tokenizer.decode(tokenizer.encode(text)))

&nbsp;
## 2.4 Adding special context tokens

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)

In [ ]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(['<|endoftext|>', "<|unk|>"])

vocab = {token:integer for integer, token in enumerate(all_tokens)}

In [ ]:
len(vocab.items())

In [ ]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

#### Tokenizer version 2

In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])

        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = "<|endoftext|>".join((text1, text2))
print(text)

In [ ]:
tokenizer.encode(text)

In [ ]:
tokenizer.decode(tokenizer.encode(text))

&nbsp;
## 2.5 BytePair encoding

In [ ]:
import importlib
import tiktoken

print("tikton version:", importlib.metadata.version("tiktoken"))

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
text =(
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

In [ ]:
strings = tokenizer.decode(integers)
print(strings)

&nbsp;
#### <|endoftext|> is last token in the gpt2 tokenizer (50256)

In [ ]:
tokenizer.n_vocab

## 2.6 Data sampling with a sliding window

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

In [ ]:
enc_sample = enc_text[50:]

In [ ]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1: context_size+1]

print(f"x:{x}")
print(f"y:     {y}")

In [ ]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "----->", desired)

In [ ]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "----->", tokenizer.decode([desired]))

In [ ]:
import torch

print("Pytorch version:", torch.__version__)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i+1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                        stride=128, shuffle=True, drop_last=True,
                        num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
        )
    
    return dataloader


In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)

first_batch = next(data_iter)
print(first_batch)

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Input:\n", inputs)
print("\nTargets:\n", targets)

&nbsp;
## 2.7 Creating token embeddings

In [ ]:
input_ids =  torch.tensor([2,3,4,5])

In [ ]:
vocab_size = 6 
ouput_dim = 3

torch.manual_seed(123)
embedding_layer =  torch.nn.Embedding(vocab_size, ouput_dim)

In [ ]:
print(embedding_layer.weight)

In [ ]:
print(embedding_layer(torch.tensor([3])))

In [ ]:
print(embedding_layer(input_ids))

&nbsp;
## 2.8 Encoding word positions

In [ ]:
vocab_size = 50257
ouput_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, ouput_dim)

In [ ]:
max_length = 4

dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [ ]:
print("Token IDs:\n", inputs)
print("\n Inputs shape:", inputs.shape)

In [ ]:
token_embedding = token_embedding_layer(inputs)
print(token_embedding.shape)
print(token_embedding)

In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go

def plot_3d_tensor(my_tensor):
    # 1. Ensure tensor is safely converted to a numpy array (handles GPU/Gradients)
    tensor_np = my_tensor.detach().cpu().numpy()

    # 2. Dynamically get dimensions and build coordinate grids
    x_dim, y_dim, z_dim = tensor_np.shape
    X, Y, Z = np.meshgrid(
        np.arange(x_dim), 
        np.arange(y_dim), 
        np.arange(z_dim), 
        indexing='ij'
    )

    # 3. Create the Volume plot
    fig = go.Figure(data=go.Volume(
        x=X.flatten(),
        y=Y.flatten(),
        z=Z.flatten(),
        value=tensor_np.flatten(),
        isomin=float(tensor_np.min()),
        isomax=float(tensor_np.max()),
        opacity=0.15,      # Adjust this to see deeper into the volume
        surface_count=15,  # Increase for more detail, decrease for better performance
        colorscale='Viridis'
    ))

    fig.update_layout(title=f"Volume Plot of Tensor Shape {tensor_np.shape}")
    fig.show()

# Example usage:
plot_3d_tensor(token_embedding)

In [ ]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, ouput_dim)

print(pos_embedding_layer.weight)

In [ ]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

print(pos_embeddings)

In [ ]:
pos_embedding_layer

In [ ]:
input_embedding = token_embedding + pos_embeddings
print(input_embedding.shape)

print(input_embedding)